# Series
## Exercise 002

Generate 10 test scores between 40 and 60, again using an index starting with September and ending with June. Find the mean of the scores and add the difference between the mean and 80 to each of the scores.

## Workout
### performing operations on series
When performing an operation on two different series, the indexes are matched, and the operation is performed via the indexes. as schown inthe following figure.
 ```
 s1 = Series([10, 20, 30, 40])
 s2 = Series([100, 200, 300, 400])
 s1 + s2
 ```
The result will be:
 
<pre>
0   110  
1   220  
2   330  
3   440
dtype: int64  
</pre> 
 
![addition-of-2-seroies](./add-series-at-the-same-index.png)

What happens if we set an explicit index rather than rely on the default positional index?
```
s1 = Series([10, 20, 30, 40], index=list('abcd'))
s2 = Series([100, 200, 300, 400], index=list('dcba'))
s1+s2
```
The result is
<pre>
a 410
b 320
c 230
d 140
dtype: int64
</pre>

![Adding 2 Series using explicit indexes](add-series-using-different-indexes.png)

If we try to add not a series and a scalar value, Pandas does something known as `broadcasting`: it applies the operator and that scalar value
to each value in the series, returning a new series.

For example:
```
s = Series([10, 20, 30, 40], index=list('abcd'))
s + 3
```
The result is
<pre>
a 13
b 23
c 33
d 43
dtype: int64
</pre>
![](./add-a-scalar-to-a-serie.png)

This operator, including comparison operators:
```
s = Series([10, 20, 30, 40], index=list('abcd'))
s < 30
```
The result is
<pre>
a     True
b     True
c    False
d    False
dtype: bool
</pre> 

In [9]:
# Solution
import numpy as np
from pandas import Series
g = np.random.default_rng()
months = 'Sep Oct Nov Dec Jan Feb Mar Apr May Jun'.split()
s = Series(g.integers(40, 61, 10), index = months)
print(f"Grades:\n{s},\n\nAugmented Grades:\n{s+(80-s.mean())}")


Grades:
Sep    41
Oct    49
Nov    42
Dec    49
Jan    58
Feb    41
Mar    45
Apr    57
May    53
Jun    43
dtype: int64,

Augmented Grades:
Sep    73.2
Oct    81.2
Nov    74.2
Dec    81.2
Jan    90.2
Feb    73.2
Mar    77.2
Apr    89.2
May    85.2
Jun    75.2
dtype: float64


There’s at least one other way to scale test scores: by looking at both the mean of the scores and their standard deviation.
Anyone who scored within one standard deviation of the mean got a C (below the mean) or a B (above the mean).
Anyone who scored more than one standard deviation above the mean got an A, and anyone who got more than one standard deviation below the mean got a D. 
During which months did our student get an A, B, C, and D?

![](./exercise-2-bis.png)


In [ ]:
# Solution
import numpy as np
import math
import pandas as pd

s = pd.Series(g.integers(40, 61, 10), index = months)
µ = s.mean()
var = ((s - µ) ** 2).sum() / s.count()
std = math.sqrt(var)

print(f"s:\n{s}\n\nVariance µ: {µ}, s.std(): {s.std(ddof=0)}, sqrt(var): {std}")

def get_grade(score: int):
    """
    Anyone who scored within one standard deviation of the mean got a C (below the mean) or a B (above the mean).
    Anyone who scored more than one standard deviation above the mean got an A, 
    and anyone who got more than one standard deviation below the mean got a D.
    A: score greater than the mean + 1 standard deviation
    B: score greater than the mean but less than or equal to the mean + 1 standard deviation
    C: score greater than or equal to the mean - 1 standard deviation but less than or equal to the mean
    D: score less than the mean - 1 standard deviation

        D< µ - std < µ < µ + std < A
              |            |
              C            B      
    This function returns:
    'A': score more than one standard deviation & above the mean
    'B': score within one standard deviation & above the mean
    'C': score within one standard deviation below the mean
    'D': score more than one standard deviation & below the mean
    '' : otherwise
    """
    # Get the statistics for the score
    if score > µ + std:
        return 'A'
    elif score > µ:
        return 'B'
    elif score >= µ - std:
        return 'C'
    else:
        return 'D'

s2 = pd.concat([s, s.apply(get_grade)], axis=1)
s2



s:
Sep    48
Oct    56
Nov    58
Dec    58
Jan    50
Feb    47
Mar    42
Apr    41
May    51
Jun    45
dtype: int64

Variance µ: 49.6, s.std(): 5.885575587824865, sqrt(var): 5.885575587824865


,0,1
Sep,48,C
Oct,56,A
Nov,58,A
Dec,58,A
Jan,50,B
Feb,47,C
Mar,42,D
Apr,41,D
May,51,B
Jun,45,C


Were there any test scores more than two standard deviations above or below the mean? If so, in which months?

In [ ]:
# Solution
# scores more than two standard deviations above or below the mean
print(f"µ-2*std = {µ-2*std} , µ+2*std = {µ+2*std}")
for month, score in s.items():
    if score > µ + 2 * std or score < µ - 2 * std:
        print(month, score)        

print(f"{s[(s > µ + 2 * std) | (s < µ - 2 * std)]}")

µ-2*std = 37.828848824350274 , µ+2*std = 61.37115117564973
Series([], dtype: int64)


* How close are the mean and median to each another?
  The mean and median are close when the data is fairly symmetric and has no strong outliers. 
  In that case, both numbers describe the “center” of the data pretty well.

* What does it mean if they are close?
  If the mean and median are close, the distribution is usually balanced around the center.
  That suggests the data is not being pulled much by extreme high or low values.

* What would it mean if they were far apart?
  If they are far apart, the data is likely skewed or contains outliers. 
  In that case, the mean gets pulled toward the extreme values more than the median does, so the median often gives a better picture of the typical value

In [18]:
#Solution
print(f"mean: {s.mean()}, median: {s.median()}, mean - median: {s.mean() - s.median()}")

mean: 49.6, median: 49.0, mean - median: 0.6000000000000014
